# Build an LLM from Scratch

## Tokenising Text

### Loading the Text

In [1]:
import urllib.request
from pathlib import Path

TEXT_FILE_LOC = Path("./the-verdict.txt")
TEXT_URL = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not TEXT_FILE_LOC.exists():
    urllib.request.urlretrieve(TEXT_URL, TEXT_FILE_LOC)

In [2]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    TEXT = f.read()

print(f"Length of text (should be 20479): {len(TEXT)}")

Length of text (should be 20479): 20479


### Generating the Text Corpus

In [3]:
import re

re_split = re.split(r'([,.:;?_!"()\']|--|\s)', TEXT)
words = [item.strip() for item in re_split if item.strip() != ""]

In [4]:
print(f"Number of tokens: {len(words)}")
print("\nExample tokens:")
print(words[:10])

Number of tokens: 4690

Example tokens:
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius']


### Converting Tokens to Token IDs

In [5]:
# Remove duplicates
words = list(sorted(set(words)))
VOCAB_SIZE = len(words)

print(f"Number of tokens: {VOCAB_SIZE}")

Number of tokens: 1130


In [6]:
vocab = {
    token: integer
    for integer, token in enumerate(words)
}

print("Vocab:")

for token, integer in tuple(vocab.items())[20:30]:
    print(f"ID {integer} --> {token}")

Vocab:
ID 20 --> Begin
ID 21 --> Burlington
ID 22 --> But
ID 23 --> By
ID 24 --> Carlo
ID 25 --> Chicago
ID 26 --> Claude
ID 27 --> Come
ID 28 --> Croft
ID 29 --> Destroyed


### Tokenising

In [7]:
class SimpleTokeniserV1:
    def __init__(self, vocab: dict[str, int]) -> None:
        self.str_to_int: dict[str, int] = vocab
        self.int_to_str: dict[int, str] = {i: s for s, i in vocab.items()}

    def encode(self, text: str) -> list[int]:
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip()
            for item in preprocessed
            if item.strip() != ""
        ]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids: list[int]) -> str:
        text = " ".join([self.int_to_str[i] for i in ids])
        return re.sub(r'\s+([,.?!"()\'])', r'\1', text)

In [8]:
tokeniser = SimpleTokeniserV1(vocab=vocab)
sample_text = "It's the last he painted, you know, Mrs. Gisburn said with pardonable pride."
sample_ids = tokeniser.encode(sample_text)
sample_decode = tokeniser.decode(sample_ids)

print("INPUT:")
print(sample_text)
print()
print("TOKENS:")
print(sample_ids)
print()
print("DECODED TOKENS:")
print(sample_decode)

INPUT:
It's the last he painted, you know, Mrs. Gisburn said with pardonable pride.

TOKENS:
[56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 67, 7, 38, 851, 1108, 754, 793, 7]

DECODED TOKENS:
It' s the last he painted, you know, Mrs. Gisburn said with pardonable pride.


### Adding Special Context Tokens 

In [9]:
try:
    sample_text = "This text will not be convered by the tokeniser!"
    sample_ids = tokeniser.encode(sample_text)
    sample_decode = tokeniser.decode(sample_ids)
except KeyError:
    print("Failed because the vocab does not have these tokens!")

Failed because the vocab does not have these tokens!


In [10]:
words.extend(["<|unk|>", "<|endoftext|>"])

vocab = {
    token: integer
    for integer, token in enumerate(words)
}

In [11]:
class SimpleTokeniserV2:
    def __init__(self, vocab: dict[str, int]) -> None:
        self.str_to_int: dict[str, int] = vocab
        self.int_to_str: dict[int, str] = {i: s for s, i in vocab.items()}

    def encode(self, text: str) -> list[int]:
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip()
            for item in preprocessed
            if item.strip() != ""
        ]
        preprocessed = [
            item
            if item in self.str_to_int else "<|unk|>"
            for item in preprocessed
        ]
        print(preprocessed)
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids: list[int]) -> str:
        text = " ".join([self.int_to_str[i] for i in ids])
        return re.sub(r'\s+([,.?!"()\'])', r'\1', text)

In [12]:
tokeniser = SimpleTokeniserV2(vocab=vocab)
sample_text = "This text will not be convered by the tokeniser!"
sample_ids = tokeniser.encode(sample_text)
sample_decode = tokeniser.decode(sample_ids)

print("INPUT:")
print(sample_text)
print()
print("TOKENS:")
print(sample_ids)
print()
print("DECODED TOKENS:")
print(sample_decode)

['This', '<|unk|>', '<|unk|>', 'not', 'be', '<|unk|>', 'by', 'the', '<|unk|>', '!']
INPUT:
This text will not be convered by the tokeniser!

TOKENS:
[97, 1130, 1130, 711, 198, 1130, 241, 988, 1130, 0]

DECODED TOKENS:
This <|unk|> <|unk|> not be <|unk|> by the <|unk|>!


### Byte pair encoding

This addresses the shortcomings of the above tokeniser, with the unknowns. There will be IDs for words that were not seen during training.

In [13]:
import tiktoken

tokeniser = tiktoken.get_encoding("gpt2")
sample_text = "Hello world"
sample_ids = tokeniser.encode(sample_text)
sample_decode = tokeniser.decode(sample_ids)

print("INPUT:")
print(sample_text)
print()
print("TOKENS:")
print(sample_ids)
print()
print("DECODED TOKENS:")
print(sample_decode)

INPUT:
Hello world

TOKENS:
[15496, 995]

DECODED TOKENS:
Hello world


In [14]:
sample_text = "Hello, do you like tea? <|endoftext|> I the sunlit terraces of some unknownPlace."

try:
    sample_ids = tokeniser.encode(sample_text)
    sample_decode = tokeniser.decode(sample_ids)
except ValueError as e:
    print(e)

Encountered text corresponding to disallowed special token '<|endoftext|>'.
If you want this text to be encoded as a special token, pass it to `allowed_special`, e.g. `allowed_special={'<|endoftext|>', ...}`.
If you want this text to be encoded as normal text, disable the check for this token by passing `disallowed_special=(enc.special_tokens_set - {'<|endoftext|>'})`.
To disable this check for all special tokens, pass `disallowed_special=()`.



There is an interesting feature here. The end of text token is usually used to concat text. It signals where one piece of text ends and where one begins.

In [15]:
sample_text = "Hello, do you like tea? <|endoftext|> I the sunlit terraces of some unknownPlace."
sample_ids = tokeniser.encode(sample_text, allowed_special={"<|endoftext|>"})
sample_decode = tokeniser.decode(sample_ids)

print("INPUT:")
print(sample_text)
print()
print("TOKENS:")
print(sample_ids)
print()
print("DECODED TOKENS:")
print(sample_decode)

INPUT:
Hello, do you like tea? <|endoftext|> I the sunlit terraces of some unknownPlace.

TOKENS:
[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 314, 262, 4252, 18250, 8812, 2114, 286, 617, 6439, 27271, 13]

DECODED TOKENS:
Hello, do you like tea? <|endoftext|> I the sunlit terraces of some unknownPlace.


### Data Sampling with a Sliding Window

The LLM cannot take in all the tokens all at once. Smaller chunks are needed, so that the LLM can be trained efficiently.

In [16]:
ENCODE_TEXT = tokeniser.encode(TEXT)

print(f"Length of the encoded text (should be 5145): {len(ENCODE_TEXT)}")

Length of the encoded text (should be 5145): 5145


In [17]:
context_size = 4

for i in range(1, context_size + 1):
    context = ENCODE_TEXT[:i]
    desired = ENCODE_TEXT[i]
    print(f"{tokeniser.decode(context)} ---> {tokeniser.decode([desired])}")

print()

for i in range(1, context_size + 1):
    context = ENCODE_TEXT[:i]
    desired = ENCODE_TEXT[i]
    print(f"{context} ---> {desired}")

I --->  H
I H ---> AD
I HAD --->  always
I HAD always --->  thought

[40] ---> 367
[40, 367] ---> 2885
[40, 367, 2885] ---> 1464
[40, 367, 2885, 1464] ---> 1807


In [18]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(DataLoader):
    def __init__(self, text: str, tokeniser, max_length: int, stride: int) -> None:
        self.input_ids: list[torch.Tensor] = []
        self.target_ids: list[torch.Tensor] = []

        token_ids = tokeniser.encode(text, allowed_special={"<|endoftext|>"})

        # Use sliding window to chunk into overlapping sequences of max_length
        for i in range(0, len(token_ids), stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self) -> int:
        return len(self.input_ids)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.input_ids[idx], self.target_ids[idx]

In [19]:
dataset = GPTDatasetV1(text=TEXT, tokeniser=tokeniser, max_length=256, stride=128)
dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)

In [20]:
first_batch = next(iter(dataloader))
torch.manual_seed(42)

print(f"First batch 1st input: {first_batch[0][0][:10]}...")
print(f"First batch 1st target: {first_batch[1][0][:10]}...\n")

print(f"First batch 2nd input: {first_batch[0][1][:10]}...")
print(f"First batch 2nd target: {first_batch[1][1][:10]}...\n")

First batch 1st input: tensor([18560,   438,  7091,   750,   523,   765,   683,   705, 28060,     6])...
First batch 1st target: tensor([  438,  7091,   750,   523,   765,   683,   705, 28060,     6,   416])...

First batch 2nd input: tensor([  262,  1633,   286, 24380,   329, 20728,   287,   257,  4286, 10273])...
First batch 2nd target: tensor([ 1633,   286, 24380,   329, 20728,   287,   257,  4286, 10273,   438])...



### Creating Token Embeddings

In [21]:
sample_ids = torch.tensor([2, 3, 5, 1])
sample_vocab_size = 6
sample_output_dim = 3

torch.manual_seed(123)
sample_embedding_layer = torch.nn.Embedding(
    num_embeddings=sample_vocab_size,
    embedding_dim=sample_output_dim
)

print(f"Shape of embedding: {sample_embedding_layer}")
print("Embedding weights:")
print(sample_embedding_layer.weight)

Shape of embedding: Embedding(6, 3)
Embedding weights:
Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [22]:
sample_embedding_layer(torch.tensor([3]))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

This is the same as row 3 (zero indexed) above! So the embedding layer is just a mapping from token IDs to vectors in the embedding matrix.

### Encoding Work Positions

In [23]:
torch.manual_seed(42)
output_dim = 256

token_embedding_layer = torch.nn.Embedding(
    num_embeddings=tokeniser.n_vocab,
    embedding_dim=output_dim
)

dataset = GPTDatasetV1(text=TEXT, tokeniser=tokeniser, max_length=4, stride=128)
dataloader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)

In [24]:
inputs, targets = next(iter(dataloader))

print(f"Token IDs:\n{inputs}")
print(f"\nInputs shape:\n{inputs.shape}")

Token IDs:
tensor([[10197,   832,   262, 46475],
        [  520,  5493,  6776,   878],
        [   40,   367,  2885,  1464],
        [  286,  1762,    30,  2011],
        [  286,   616,  4286,   705],
        [ 1021,   757,   438, 10919],
        [  262,  1633,   286, 24380],
        [  198,  1544, 13818,  4622]])

Inputs shape:
torch.Size([8, 4])


In [25]:
token_embeddings = token_embedding_layer(inputs)
print(f"Token embeddings shape: {token_embeddings.shape}")

Token embeddings shape: torch.Size([8, 4, 256])


This means we have `8` sets of embeddings in a batch, with `4` embedded tokens inside. Each embedding is `256` in length.

There's now the problem that if we have the phrase "the fox jumps over the fox", they both instances of "fox" would map onto the same embedding vector. Positional information can encode the differences between them.
The idea here is that positional information embedded and then is simply added onto the embedded tokens.

In [26]:
context_length = 4
pos_embedding_layer = torch.nn.Embedding(
    context_length,
    embedding_dim=output_dim,
)
print(f"Positional vector to be embedded and added onto the embeddigs: {torch.arange(context_length)}")

Positional vector to be embedded and added onto the embeddigs: tensor([0, 1, 2, 3])


In [27]:
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(f"Shape of positional embeddings layer: {pos_embeddings.shape}")

Shape of positional embeddings layer: torch.Size([4, 256])


In [28]:
input_embeddings = token_embeddings + pos_embeddings
print(f"Input shape of embeddings going into the LLM: {input_embeddings.shape}")

Input shape of embeddings going into the LLM: torch.Size([8, 4, 256])


PyTorch here add the positonal embeddings to each batch separately.

### Conclusion

This covers the entire input pipeline. To summerise the steps:

1. Gather input text.
2. Tokenise the text. This means to break down each part of the text into indiviual bits of text.
3. Convert each token to token IDs.
4. Map each token ID to an embedding.
5. Add on the positional embedding information.